# Week 8: Comprehensive Model Evaluation and Price-Band Analysis

Evaluate the saved Weeks 4-7 models with consistent metrics and training-derived price bands. All results are generated from authorized local data. Do not publicly share executed outputs or licensed records.

## 1. Evaluation protocol

December 2025-March 2026 trains; April validates; May tests. Reuse every saved pipeline without fitting or changing features. All-cleaned sales are the primary reliability population; the original training-target bounds are a separate diagnostic population. Freeze the April assessment before generating May predictions. May was previously examined and is not an untouched holdout.

MAE = mean absolute residual; RMSE = square root of mean squared residual; R2 = 1 - SSE/SST. MAPE and MdAPE are the mean and median of 100 times absolute residual divided by positive actual price. Signed bias is prediction minus actual. Nonfinite predictions fail evaluation. Invalid actuals and nonpositive percentage denominators have explicit counts. Price-band R2 requires at least 50 observations and nonconstant actuals.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import pandas as pd
from IPython.display import display, Image, Markdown
from src.week8 import run
from src.week8_report import CHARTS
OUT = ROOT / 'outputs' / 'week08'

## 2. Execute comprehensive evaluation

The runner audits model schemas, reproduces historical metrics, evaluates full chronological partitions, and generates results and figures. It writes only new Week 8 artifacts and the Week 8 report, preserving prior weekly results. No new dependencies or model training are needed.

In [ ]:
summary, manifest = run(ROOT)
print('Saved pipelines:', manifest['model_count'])
print('Consolidated metric rows:', len(summary))
print('Historical reproduction checks:', manifest['historical_reproduction_checks'])
assert manifest['refitted_models'] == 0
assert manifest['may_used_for_selection'] is False

## 3. Verify what actually exists

The inventory includes the baseline, three Week 5 family winners, six Week 6 engineered/district pipelines, and the saved Week 7 removal winner. Metrics-only candidates are archived separately; no boosting model is invented or inferred.

In [ ]:
inventory = pd.read_csv(OUT / 'model_inventory.csv')
display(inventory[['model_id','estimator','predictors','artifact']])
display(pd.read_csv(OUT / 'archived_experiments.csv'))

## 4. Examine the training distribution and define categories

Use all cleaned training prices to learn Q25, Q50 and Q90. Lower-priced, mid-range, upper-mid-range and luxury correspond approximately to 25%, 25%, 40% and 10% of training sales. Ties may change proportions. Luxury is a relative top-decile label. Intervals are lower inclusive and upper exclusive, with unbounded outer intervals. Exact boundaries are stored in price_bands.json; these retrospective categories never enter the model.

In [ ]:
display(pd.read_csv(OUT / 'training_price_distribution.csv'))
print('Exact category boundaries:', manifest['price_bands']['cuts'])
display(Image(filename=str(OUT / 'figures' / 'training_price_distribution.png')))

## 5. April validation comparison

Identify separate leaders for R2, MAE and MdAPE. The balanced assessment uses equal mean ranks of those three metrics on all-cleaned April, with model ID breaking rank ties. RMSE and MAPE remain reported. A rank does not establish a meaningful or statistically significant difference. Bounded rankings are a secondary comparison, and May cannot alter either assessment.

In [ ]:
assessment = pd.read_csv(OUT / 'validation_comparison.csv')
for population in ['all_cleaned','bounded']:
    print('April validation:', population)
    display(assessment.loc[assessment.population.eq(population), ['model_id','n','R2','MAE_USD','RMSE_USD','MAPE_pct','MdAPE_pct','balanced_rank']].sort_values('balanced_rank'))
    print('Metric leaders:', manifest['selection']['leaders'][population])
selected = manifest['selection']['selected_model_id']
print('April all-cleaned balanced choice:', selected)

## 6. May test results, reported separately

These are descriptive comparisons of already frozen models. May was already inspected in earlier work. Do not use this table to select a model, engineer features or tune parameters.

In [ ]:
for population in ['all_cleaned','bounded']:
    print('May test:', population)
    display(summary.loc[summary.split.eq('test') & summary.population.eq(population) & summary.price_band.eq('All'), ['model_id','n','R2','MAE_USD','RMSE_USD','MAPE_pct','MdAPE_pct','bias_USD']])

## 7. Price-band reliability and prediction bias

Show all required metrics and counts for the April balanced choice on both months and both populations. The consolidated CSV contains these breakdowns for every saved model. Positive mean bias means overprediction; negative means underprediction. Negative subgroup R2 can occur even when overall R2 is positive because each band has a much narrower price range.

In [ ]:
for split in ['validation','test']:
    for population in ['all_cleaned','bounded']:
        print(split, population, selected)
        display(summary.loc[summary.model_id.eq(selected) & summary.split.eq(split) & summary.population.eq(population) & summary.price_band.ne('All'), ['price_band','n','MAE_USD','RMSE_USD','MAPE_pct','MdAPE_pct','R2','R2_reason','bias_USD']])
display(pd.read_csv(OUT / 'tail_sensitivity.csv').query('model_id == @selected'))

## 8. Visual diagnostics

All seven requested chart types are exported as PNG and PDF for both months and populations. Display all-cleaned charts below; change VIEW_POPULATION to bounded to inspect the earlier restricted population. Full price/error ranges are retained with labeled logarithmic or symmetric-log axes. Histogram heights count sales per unequal-width bin. Band charts compare the April choice with the original baseline and Week 6 reference.

In [ ]:
VIEW_POPULATION = 'all_cleaned'
for split in ['validation','test']:
    for chart in CHARTS:
        display(Image(filename=str(OUT / 'figures' / f'{split}_{VIEW_POPULATION}_{chart}.png')))

## 9. Executed findings, limitations and recommendations

Read the generated report. Interpret changes on reused April validation as exploratory. Tail prices, sold-snapshot attributes and retrospective district boundaries limit reliability. Use later unseen months and spatial holdouts for confirmation.

In [ ]:
display(Markdown((ROOT / 'docs' / 'WEEK8_RESULTS.md').read_text(encoding='utf-8')))

## 10. Verify every output

Recompute consolidated metrics from saved prediction arrays; check band counts, fixed April selection, historical reproduction, valid figures and all input/output hashes. No refitting occurs.

In [ ]:
from scripts.verify_week8 import verify
verified = verify(ROOT)
assert verified['historical_files_unchanged'] is True